# Laboratorio 1 - Preparación de un corpus y EDA

Notebook paso a paso, con celdas de explicación y comentarios en el código.

**Corpus:** Spanish News Classification  

## 1. Objetivo

Explorar el corpus, aplicar el pipeline de normalización visto en clase y responder las preguntas de EDA y análisis.

In [ ]:
# Importaciones
from collections import Counter
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk.stem.snowball import SpanishStemmer
from spacy.lang.es.stop_words import STOP_WORDS as ES_STOPWORDS
from wordcloud import WordCloud

# Rutas del proyecto
INPUT = Path('df_total.csv') # Ruta del archivo de entrada que contiene el DataFrame con los datos
OUTDIR = Path('lab1_nlp_final') # Ruta del directorio de salida donde se guardarán los resultados
OUTDIR.mkdir(exist_ok=True)

In [2]:
# Cargar el corpus
df = pd.read_csv(INPUT)
df.head()

,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encaden...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión n...,Otra


## 2. Explorar el corpus antes de tocarlo

Preguntas: cuántos documentos hay, qué columnas contiene, cómo se distribuyen las categorías y si hay vacíos o duplicados.

In [3]:
# Resumen básico del corpus
print("Documentos:", len(df))
print("Columnas:", df.columns.tolist())
print("\nVacíos por columna:\n", df.isna().sum())
print("\nFilas duplicadas:", int(df.duplicated().sum()))
print("\nDistribución de categorías:\n", df["Type"].value_counts())

Documentos: 1217
Columnas: ['url', 'news', 'Type']

Vacíos por columna:
 url     0
news    0
Type    0
dtype: int64

Filas duplicadas: 75

Distribución de categorías:
 Type
Macroeconomia     340
Alianzas          247
Innovacion        195
Regulaciones      142
Sostenibilidad    137
Otra              130
Reputacion         26
Name: count, dtype: int64


### Respuestas de exploración

- El corpus tiene **1217 documentos**.
- Las columnas son **url**, **news** y **Type**.
- `url` guarda el enlace original, `news` guarda el texto completo y `Type` la categoría.
- Se detectaron **0 filas vacías** y **75 filas duplicadas**.
- La distribución de categorías aparece en la tabla siguiente.

In [4]:
# Distribución de categorías
category_counts = df["Type"].value_counts()
category_counts

Type
Macroeconomia     340
Alianzas          247
Innovacion        195
Regulaciones      142
Sostenibilidad    137
Otra              130
Reputacion         26
Name: count, dtype: int64

## 3. Preparación del corpus

Se aplica el pipeline en este orden: tokenización, minúsculas, eliminación de puntuación, eliminación de stopwords y lematización.

Nota: como no está instalado un modelo español completo de spaCy en este entorno, la última etapa se implementa como **stemming** con `SpanishStemmer` como aproximación.

In [5]:
# Configuración del preprocesamiento
punct_re = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)
word_re = re.compile(r"^\w+$", flags=re.UNICODE)
stopwords = set(ES_STOPWORDS) | {"rt", "http", "https", "www", "com"}
stemmer = SpanishStemmer()

# Tokenizar: divide cada texto en palabras y signos
def tokenize(text):
    return punct_re.findall(str(text))

# Pasar a minúsculas
def to_lower(tokens):
    return [t.lower() for t in tokens]

# Quitar puntuación
def remove_punct(tokens):
    return [t for t in tokens if word_re.match(t)]

# Quitar stopwords
def remove_stop(tokens):
    return [t for t in tokens if t not in stopwords]

# Normalización final: stemming como aproximación a lematización
def normalize_tokens(text):
    tokens = tokenize(text)
    tokens = to_lower(tokens)
    tokens = remove_punct(tokens)
    tokens = remove_stop(tokens)
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

In [6]:
# Medir tokens y tipos después de cada paso
def flatten(series):
    return [t for doc in series for t in doc]

stage_tokens = {
    "Tokenización": flatten(df["news"].apply(tokenize)),
    "Minúsculas": flatten(df["news"].apply(lambda x: to_lower(tokenize(x)))),
    "Eliminar puntuación": flatten(df["news"].apply(lambda x: remove_punct(to_lower(tokenize(x))))),
    "Eliminar stopwords": flatten(df["news"].apply(lambda x: remove_stop(remove_punct(to_lower(tokenize(x)))))),
    "Lematización (fallback a stemming)": flatten(df["news"].apply(normalize_tokens)),
}

stage_summary = pd.DataFrame([
    {"Paso": stage, "Tokens": len(tokens), "Tipos": len(set(tokens))}
    for stage, tokens in stage_tokens.items()
])
stage_summary["Reducción vs. tokenización"] = (
    (stage_summary.loc[0, "Tipos"] - stage_summary["Tipos"]) / stage_summary.loc[0, "Tipos"] * 100
).round(2)
stage_summary

,Paso,Tokens,Tipos,Reducción vs. tokenización
0,Tokenización,684297,32606,0.00
1,Minúsculas,684297,29571,9.31
2,Eliminar puntuación,637564,29530,9.43
3,Eliminar stopwords,297489,29073,10.84
4,Lematización (fallback a stemming),297489,15101,53.69


## 4. ¿Qué es un EDA?

EDA significa Análisis Exploratorio de Datos. Su objetivo es entender la estructura del corpus, detectar patrones y revisar calidad antes de modelar.